## Llama 3.1 8B zero-shot structured-output

This notebook uses Outlines and the repository's current prompt and Pydantic schema to extract structured valid JSON from the synthetic free-text rectal MRI reports.

In [ ]:
import json
import sys
import time
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'llm_benchmark_example':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from llm_benchmark_example.prompt import SYSTEM_PROMPT
from llm_benchmark_example.pydantic_template import RectalCancerReport

In [ ]:
DATA_PATH = PROJECT_ROOT / 'generated_datasets' / 'updated_synthetic_dataset.csv'
OUTPUT_DIR = PROJECT_ROOT / 'generated_datasets' / 'llm_benchmark_outputs'

df = pd.read_csv(DATA_PATH)
required_columns = {'structured_report', 'free_text_report'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required dataset columns: {sorted(missing_columns)}')

print(f'Loaded {len(df):,} reports from {DATA_PATH}')

## Inference environments

The original inference was run on premises in a Docker container using an NVIDIA RTX A6000 GPU and a local Llama 3.1 8B Instruct checkpoint.

### Google Colab

A similar experiment can be run in Google Colab:

1. Select a GPU runtime under **Runtime > Change runtime type**.
2. Accept the Meta Llama 3.1 model access terms on Hugging Face.
3. Add a Colab secret named `HF_TOKEN` with a Hugging Face token that can access the model.
4. Uncomment and run the setup cell below.

The Colab path loads `meta-llama/Llama-3.1-8B-Instruct` with 4-bit BitsAndBytes quantization and automatic device placement to reduce GPU-memory use. The on-premises path retains full-precision loading from the local checkpoint. CUDA diagnostics and model loading remain intentionally commented out so that opening or running the notebook does not allocate a GPU or download model weights automatically.

In [ ]:
# Google Colab only: uncomment and run this once after selecting a GPU runtime.
# %pip install -q -U "outlines[transformers]>=1.0" "transformers>=4.56" "accelerate>=0.33" "bitsandbytes>=0.43" "huggingface-hub>=0.24"

# import torch
# from outlines import Generator, from_transformers
# from transformers import AutoModelForCausalLM, AutoTokenizer

# RUNNING_IN_COLAB = 'google.colab' in sys.modules
# if not torch.cuda.is_available():
#     raise RuntimeError('A CUDA GPU is required. In Colab, select a GPU runtime first.')

# print(f'CUDA device: {torch.cuda.get_device_name(0)}')
# compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
# model_kwargs = {
#     'dtype': compute_dtype,
#     'low_cpu_mem_usage': True,
# }

# if RUNNING_IN_COLAB:
#     from google.colab import userdata
#     from huggingface_hub import login
#     from transformers import BitsAndBytesConfig

#     hf_token = userdata.get('HF_TOKEN')
#     login(token=hf_token)
#     MODEL_NAME_OR_PATH = 'meta-llama/Llama-3.1-8B-Instruct'
#     model_kwargs.update({
#         'token': hf_token,
#         'device_map': 'auto',
#         'quantization_config': BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type='nf4',
#             bnb_4bit_use_double_quant=True,
#             bnb_4bit_compute_dtype=compute_dtype,
#         ),
#     })
# else:
#     hf_token = None
#     MODEL_NAME_OR_PATH = '/workspace/models_params/llama/Llama-3.1-8B-Instruct'

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME_OR_PATH, token=hf_token)
# hf_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME_OR_PATH,
#     **model_kwargs,
# )
# if not RUNNING_IN_COLAB:
#     hf_model = hf_model.to('cuda:0')
# hf_model.eval()

# outlines_model = from_transformers(hf_model, tokenizer)
# report_processor = Generator(outlines_model, RectalCancerReport)

In [ ]:
SCHEMA = json.dumps(RectalCancerReport.model_json_schema(), indent=2)
MAX_NEW_TOKENS = 2048

def build_zero_shot_prompt():
    return SYSTEM_PROMPT.format(schema=SCHEMA)

def format_chat(system_prompt, report):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': f'REPORT: {report}\n\n### OUTPUT:'},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [ ]:
def parse_structured_result(result):
    return RectalCancerReport.model_validate_json(result).model_dump()

def run_benchmark(input_df, system_prompt, output_column):
    if 'tokenizer' not in globals() or 'report_processor' not in globals():
        raise RuntimeError('Uncomment and run the CUDA/model-loading cell first.')

    outputs = []
    durations = []
    for position, report in enumerate(input_df['free_text_report'], start=1):
        prompt = format_chat(system_prompt, report)
        started_at = time.perf_counter()
        result = report_processor(prompt, max_new_tokens=MAX_NEW_TOKENS)
        elapsed = time.perf_counter() - started_at
        outputs.append(parse_structured_result(result))
        durations.append(elapsed)
        print(f'Processed report {position:,}/{len(input_df):,} in {elapsed:.1f} seconds')

    output_df = input_df.copy()
    output_df[output_column] = [
        json.dumps(item, ensure_ascii=False) for item in outputs
    ]
    mean_duration = sum(durations) / len(durations)
    print(f'Average inference time: {mean_duration:.1f} seconds per report')
    return output_df, durations

def save_results(output_df, filename):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / filename
    output_df.to_csv(output_path, index=False)
    print(f'Saved results to {output_path}')
    return output_path

## Zero-shot

In [ ]:
zero_shot_df, zero_shot_durations = run_benchmark(
    input_df=df,
    system_prompt=build_zero_shot_prompt(),
    output_column='llama3_1_8b_zero_shot_json',
)
zero_shot_path = save_results(
    zero_shot_df, 'llama3_1_8b_zero_shot.csv'
)